# 05 完整评分卡训练与部署

每节下方的完整代码块均可单独复制运行，包含自己的 import、数据加载和实际调用。使用全部订单，不做行采样。安装所需依赖后，从 `examples` 目录运行。

In [ ]:
import sys
sys.path.append('../')

## 1. 从数据到部署的完整流程

下方完整代码块可单独运行，依次完成全量数据切分、评分卡训练、超参数搜索、固定模型、业务报告与部署一致性检查。

In [1]:
import sys

sys.path.append("../")

import numpy as np
import pandas as pd
from IPython.display import display
from hscredit import hscredit_dataset
from hscredit.utils.datasets import HSCREDIT_EXAMPLE_COLUMNS
from pathlib import Path
from tempfile import mkdtemp
from sklearn.base import clone
from optuna.samplers import GridSampler
from hscredit import (
    OptimalBinning,
    ScoreCard,
    RoundScoreCard,
    ProbabilityScoreCard,
    LogisticRegression,
    ModelTuner,
)
from hscredit.report import ModelReport, feature_bin_stats
from hscredit.utils import save_pickle, load_pickle

from hscredit.core.model_selection import oot_split, group_train_test_split

data = hscredit_dataset(columns=HSCREDIT_EXAMPLE_COLUMNS)

features = ["BASE_X187", "DK_X1689", "DK_X44", "DK_X1995_2", "DK_X1223", "DK_X2677"]
target = "MOB6@30"
data["MOB1@30"] = data["MOB1"].gt(30).astype(int)
data["MOB3@30"] = data["MOB3"].gt(30).astype(int)
data["MOB6@30"] = data["MOB6"].gt(30).astype(int)
data["风险调整金额"] = data["loan_amt"].where(data["MOB6"].le(60), -data["loan_amt"])
data["授信时间"] = pd.to_datetime(data["授信时间"])

# 最新自然月作为 OOT；同一客户不能同时进入开发集和 OOT。
oot_start = data["授信时间"].max().to_period("M").start_time
development, oot = oot_split(data, time_col="授信时间", oot_start=oot_start)
development = development.loc[~development["用户编码"].isin(oot["用户编码"])].copy()
train, test = group_train_test_split(development, group="用户编码", test_size=0.25, random_state=42)
fit_data, validation = group_train_test_split(
    train, group="用户编码", test_size=0.25, random_state=42
)

# 原始字段进入分箱器，评分卡内部完成 WOE 和 LR。
binner = OptimalBinning(
    method="quantile", max_n_bins=4, monotonic=False, missing_separate=True, n_jobs=1
)
card = ScoreCard(
    target=target,
    binner=binner,
    lr_model=LogisticRegression(max_iter=300, calculate_stats=True),
    calculate_stats=True,
    base_score=600,
    pdo=50,
    base_odds=20,
)
card.fit(fit_data[features + [target]], input_type="raw")
display(card.scorecard_points())
display(
    feature_bin_stats(
        fit_data,
        feature="DK_X1689",
        overdue=["MOB3", "MOB6"],
        dpds=30,
        overdue_operator=">",
        binner=card.binner,
        n_jobs=1,
    )
)

# 搜索时使用未拟合的克隆，允许每次重新训练 LR 与分箱组件。
search_space = {"lr_model__C": [0.3, 2.0]}
tuner = ModelTuner(
    clone(card),
    search_space=search_space,
    metric="auc",
    cv=None,
    fit_params={"input_type": "raw"},
    n_jobs=1,
    random_state=42,
    sampler=GridSampler(search_space, seed=42),
    retention="predictions",
)
tuner.fit(
    fit_data[features + [target]],
    eval_set={"验证集": validation[features + [target]]},
    n_trials=2,
    show_progress_bar=False,
)
display(tuner.get_optimization_history())
display(tuner.plot_optimization_history())

# 固定参数，使用完整 train 重训。
best = clone(tuner.get_best_model())
best.fit(train[features + [target]], input_type="raw")
display(best.scorecard_scale())
display(best.scorecard_points())
display(best.lr_model_.summary())
display(best.get_detailed_score(test[features]).head())
display(best.get_reason_codes(test[features]).head())

# 取整评分卡和通用概率评分卡。
rounded = RoundScoreCard(scorecard=best, decimal=0)
probability_card = ProbabilityScoreCard(model=best, prefit=True, target=target)
probability_card.fit(train[features + [target]])
display(
    pd.DataFrame(
        {
            "评分卡": best.predict(test[features]),
            "取整评分": rounded.predict(test[features]),
            "概率评分卡": probability_card.predict(test[features]),
        }
    ).head()
)

datasets = {"训练集": train, "测试集": test, "OOT": oot}
report = ModelReport(
    best,
    datasets=datasets,
    features=features,
    overdue=["MOB1", "MOB3", "MOB6"],
    dpds=30,
    overdue_operator=">",
    n_jobs=1,
)
display(report.summary())
output = Path(mkdtemp(prefix="hscredit_scorecard_"))
report.to_excel(
    output / "评分卡报告.xlsx",
    amount="loan_amt",
    date="授信时间",
    with_plots=True,
    include_sample_records=False,
    mode="strict",
)

# 部署代码由 hscredit 生成，Notebook 不自行实现计分逻辑。
best.export(to_json=str(output / "评分卡规则.json"))
python_code = best.export_deployment_code(
    language="python", output_file=str(output / "评分卡.py"), function_name="calculate_score"
)
best.export_deployment_code(language="sql", output_file=str(output / "评分卡.sql"))
best.export_deployment_code(language="java", output_file=str(output / "评分卡.java"))
namespace = {}
exec(python_code, namespace)
deployment_scores = [
    namespace["calculate_score"](record) for record in test[features].to_dict("records")
]
display(
    pd.DataFrame({"模型评分": best.predict(test[features]), "部署评分": deployment_scores}).head()
)
np.testing.assert_allclose(deployment_scores, best.predict(test[features]), atol=1e-6)

restored = ScoreCard().load_rules(output / "评分卡规则.json")
np.testing.assert_allclose(
    restored.predict(test[features]), best.predict(test[features]), atol=1e-6
)
rounded.export(to_json=str(output / "取整评分卡规则.json"))
save_pickle(best, output / "完整评分卡.pkl")
print(output)

C:\Users\18306\AppData\Roaming\Python\Python314\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
C:\Users\18306\AppData\Roaming\Python\Python314\site-packages\sklearn\linear_model\_logistic.py:1160: UserWarning: Inconsistent values: penalty=l2 with l1_ratio=None. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(


,变量名称,变量含义,变量分箱,对应分数,WOE值
0,基础分,截距项（基准分数）,-,549.8050,NaN
1,BASE_X187,,"[-inf, 37)",-15.8389,0.2331
2,BASE_X187,,"[37, 51)",-6.4817,0.0954
3,BASE_X187,,"[51, 66)",2.2041,-0.0324
4,BASE_X187,,"[66, +inf)",23.4275,-0.3448
5,DK_X1689,,"[-inf, 21)",2.4530,-0.1722
6,DK_X1689,,"[21, 31)",0.8306,-0.0583
7,DK_X1689,,"[31, 43)",-0.4781,0.0336
8,DK_X1689,,"[43, +inf)",-2.9327,0.2058
9,DK_X1689,,缺失值,-310.9643,21.8242


分箱详情                                     MOB3_30+                            ... MOB6_30+                                                                   
       指标名称      指标含义        分箱标签   样本总数   样本占比     好样本数 坏样本数  好样本占比  坏样本占比   坏样本率  ...    指标IV值  LIFT值    坏账改善   风险拒绝比 累积LIFT值 累积坏账改善 累计风险拒绝比 累积好样本数 累积坏样本数  分档KS值
0  DK_X1689  DK_X1689  [-inf, 21)  10724 0.2287    10062  662 0.2261 0.2775 0.0617  ...   0.0200 1.1679  0.0498  0.2177  1.1679 0.0498  0.2177   9583   1141 0.0423
1  DK_X1689  DK_X1689    [21, 31)  11838 0.2525    11223  615 0.2522 0.2578 0.0520  ...   0.0200 1.0543  0.0183  0.0726  1.1083 0.1004  0.2087  20284   2278 0.0573
2  DK_X1689  DK_X1689    [31, 43)  12279 0.2619    11678  601 0.2625 0.2519 0.0489  ...   0.0200 0.9699 -0.0107 -0.0408  1.0595 0.1722  0.2317  31478   3363 0.0487
3  DK_X1689  DK_X1689  [43, +inf)  12037 0.2568    11529  508 0.2591 0.2129 0.0422  ...   0.0200 0.8280 -0.0594 -0.2314  1.0001 1.0000  1.0001  42607   4271 0.0001
4  DK_X1689  DK_X1689     missing      3 0.0001        3    0 0.0001 0.0000 0.0000  ...   0.0200 0.0000 -0.0001 -1.0001  1.0000 1.0000  0.0000  42610   4271 0.0000

[5 rows x 39 columns]

[I 2026-11-09 14:14:16,772] A new study created in memory with name: no-name-c06d5769-d505-474f-bedf-e2f1f0c97f1d


C:\Users\18306\AppData\Roaming\Python\Python314\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
C:\Users\18306\AppData\Roaming\Python\Python314\site-packages\sklearn\linear_model\_logistic.py:1160: UserWarning: Inconsistent values: penalty=l2 with l1_ratio=None. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(


[I 2026-11-09 14:14:17,059] Trial 0 finished with value: {'AUC': 0.5559753256509413} and parameters: {'lr_model__C': 2.0}. Best is trial 0 with value: 0.5559753256509413.


... (10 outputs left out)

,评分卡,取整评分,概率评分卡
0,531.2322,532.0000,572.0000
1,531.2322,532.0000,572.0000
2,535.1018,535.0000,574.0000
3,531.4284,532.0000,573.0000
4,531.4284,532.0000,573.0000


统计指标        KS                  AUC                      样本数                         坏样本率              
数据集        训练集    测试集    OOT    训练集    测试集    OOT        训练集        测试集        OOT    训练集    测试集    OOT
逾期指标                                                                                                   
MOB1@30 0.1656 0.1677 0.1645 0.6093 0.6109 0.5958 62516.0000 20838.0000 28025.0000 0.0143 0.0154 0.0107
MOB3@30 0.1109 0.1238 0.1028 0.5739 0.5822 0.5684 62516.0000 20838.0000 28025.0000 0.0516 0.0544 0.0450
MOB6@30 0.0944 0.0965 0.0913 0.5607 0.5626 0.5622 62516.0000 20838.0000 28025.0000 0.0909 0.0934 0.0894

,模型评分,部署评分
0,531.2322,531.2322
1,531.2322,531.2322
2,535.1018,535.1018
3,531.4284,531.4284
4,531.4284,531.4284


C:\Users\18306\AppData\Local\Temp\hscredit_scorecard_iu2g_vgb
